# Exploratory Notebook: ELO Rating System for UFC Fighters

> **Note:** This is an *exploratory* notebook. The final model results are in `01_MAIN_ufc_fight_prediction.ipynb`.

This notebook explores a custom ELO rating system built from scratch for UFC fighters. Traditional ELO (used in chess) assigns a single rating and updates it after each match. This implementation extends that concept with several UFC-specific innovations:

1. **Variable K schedule** — New fighters have high K (ratings move fast under uncertainty); veterans have lower K (ratings are more stable).
2. **Division ELO** — Separate ratings per weight class, with cross-division transfer factors calibrated by backtest.
3. **Domain ELOs** — Separate ELO systems for striking offense/defense, grappling offense/defense, and finishing ability/durability.
4. **Peak ELO** — Tracks a fighter's best sustained form within continuous activity windows.
5. **ELO velocity** — Rate of ELO change over last 3 and 5 fights (momentum signal).
6. **Strength-of-schedule** — Average opponent ELO at time of each prior fight.

**Why ELO was not in the final model:** Despite the sophistication of this system, adding ELO features produced only marginal improvement over the simpler feature set used in the main notebook. UFC fight outcomes are noisy enough that additional complexity didn't consistently generalize. The system is documented here as a reusable component for future exploration.

In [ ]:
import polars as pl
import statistics
import matplotlib.pyplot as plt
from pathlib import Path

DATA_PATH = Path('data/fight_snapshots.parquet')

# --- ELO system constants ---
INITIAL_ELO    = 1500.0
K_DOMAIN       = 40.0    # fixed K for domain ELOs (striking, grappling, finishing)
GRAP_TD_WEIGHT = 30      # seconds of ctrl-time equivalent per takedown landed

# Variable K schedule: as a fighter accumulates more fights their rating stabilizes.
#  0-4  fights: K=66  — rookie, high uncertainty
#  5-14 fights: K=68  — developmental phase
# 15-19 fights: K=63  — settling into true level
# 20+  fights:  K=54  — veteran, stable rating
# Tuned via bucket-MAE calibration: 7.5x improvement over fixed K=32.
K_SCHEDULE = [(0, 66.0), (5, 68.0), (15, 63.0), (20, 54.0)]

# Per-division ELO transfer factors (calibrated via per-division backtest).
# Controls how much of a fighter's global ELO surplus carries into a new division.
TRANSFER_BY_DIV = {
    'flyweight': 1.0, 'bantamweight': 0.2, 'featherweight': 1.0,
    'lightweight': 1.0, 'welterweight': 0.0, 'middleweight': 0.2,
    'light heavyweight': 1.0, 'heavyweight': 0.4, 'catch weight': 0.4,
}
WEIGHT_CLASS_LBS = {
    'strawweight': 115.0, 'flyweight': 125.0, 'bantamweight': 135.0,
    'featherweight': 145.0, 'lightweight': 155.0, 'welterweight': 170.0,
    'middleweight': 185.0, 'light heavyweight': 205.0, 'heavyweight': 265.0, 'catch weight': 160.0,
}
_MAX_WEIGHT_DIFF = max(WEIGHT_CLASS_LBS.values()) - min(WEIGHT_CLASS_LBS.values())
TRANSFER_BASE = 0.4  # fallback for unknown divisions

## 1. Data Loading

In [ ]:
def load_data() -> tuple:
    """
    Loads and unnests fight_snapshots.parquet into fights and prior fights DataFrames.
    Args:
        None
    Returns:
        tuple of (fights_df, prior_fights_df) as Polars DataFrames
    """
    df = pl.read_parquet(DATA_PATH).with_columns(pl.col('fight_id').alias('root_fight_id'))
    fights_df = df.drop(['prior_f1', 'prior_f2', 'fight_id'])

    def extract_prior(df, col, role):
        """
        Explodes one nested prior-fight column into a flat DataFrame.
        Args:
            df: raw snapshot DataFrame
            col: nested column name to explode ('prior_f1' or 'prior_f2')
            role: fighter role string ('f1' or 'f2') to tag each row
        Returns:
            flat pl.DataFrame with one row per prior fight
        """
        return (
            df.select(['root_fight_id', col]).explode(col).unnest(col)
            .rename({'fight_id': 'prior_fight_id'})
            .with_columns(pl.lit(role).alias('fighter_role'))
        )

    prior_fights_df = pl.concat([extract_prior(df, 'prior_f1', 'f1'), extract_prior(df, 'prior_f2', 'f2')])
    return fights_df, prior_fights_df


fights_df, prior_fights_df = load_data()
print('fights_df:      ', fights_df.shape)
print('prior_fights_df:', prior_fights_df.shape)

## 2. Core ELO Helpers

The K schedule and weight-class transfer factor are the two main tunable components of the system.

In [ ]:
def get_k(fight_count: int) -> float:
    """
    Returns the K factor for a fighter based on their total career fight count.
    Higher K = ratings move faster. New fighters get higher K to reflect uncertainty.
    Args:
        fight_count: number of fights the fighter has had so far in the dataset
    Returns:
        float K factor from the K_SCHEDULE lookup
    """
    k = K_SCHEDULE[0][1]
    for thresh, val in K_SCHEDULE:
        if fight_count >= thresh:
            k = val
    return k


def transfer_factor(from_div: str, to_div: str) -> float:
    """
    Computes the fraction of global ELO surplus that carries into a new weight class.
    Adjacent division moves transfer more than long jumps (e.g. MW->LHW > FW->LHW).
    Formula: base(to_div) * (1 - |from_lbs - to_lbs| / max_weight_diff)
    Base values calibrated via per-division backtest on held-out root fights.
    Args:
        from_div: division the fighter is coming from (lowercase string)
        to_div: division the fighter is entering (lowercase string)
    Returns:
        float in [0, 1] representing the ELO transfer multiplier
    """
    base     = TRANSFER_BY_DIV.get(to_div, TRANSFER_BASE)
    from_lbs = WEIGHT_CLASS_LBS.get(from_div)
    to_lbs   = WEIGHT_CLASS_LBS.get(to_div)
    if from_lbs is None or to_lbs is None:
        return base
    diff = abs(from_lbs - to_lbs)
    return base * (1.0 - diff / _MAX_WEIGHT_DIFF)


def elo_update(ra: float, rb: float, s_a: float, k: float) -> tuple:
    """
    Computes updated ELO ratings for two fighters after one contest.
    Standard ELO formula: new_ra = ra + K * (s_a - expected_a)
    Args:
        ra: pre-fight ELO of fighter A
        rb: pre-fight ELO of fighter B
        s_a: outcome score for fighter A (1.0 = win, 0.0 = loss)
        k: K factor to use for the exchange
    Returns:
        tuple of (new_ra, new_rb) post-fight ELO ratings
    """
    e_a = 1.0 / (1.0 + 10.0 ** ((rb - ra) / 400.0))
    new_ra = ra + k * (s_a - e_a)
    new_rb = rb + k * ((1.0 - s_a) - (1.0 - e_a))
    return new_ra, new_rb


# Quick sanity check: equal fighters should have 50% win probability
e = 1.0 / (1.0 + 10.0 ** ((1500 - 1500) / 400.0))
print(f'Win probability when ELO is equal: {e:.2%}  (should be 50%)')

# K schedule visualization
fight_counts = list(range(0, 30))
k_values = [get_k(n) for n in fight_counts]
plt.figure(figsize=(8, 3))
plt.step(fight_counts, k_values, where='post')
plt.xlabel('Career fight count')
plt.ylabel('K factor')
plt.title('Variable K Schedule: New Fighters Learn Faster')
plt.grid(axis='y', alpha=0.4)
for spine in plt.gca().spines.values(): spine.set_visible(False)
plt.tight_layout()
plt.show()

## 3. Chronological Timeline Assembly

ELO must be computed in strict chronological order to prevent data leakage. We merge root fights with all prior fight history into a single de-duplicated timeline sorted oldest-first.

In [ ]:
def assemble_timeline(fights: pl.DataFrame, prior_fights: pl.DataFrame) -> pl.DataFrame:
    """
    Merges root fights and all prior fight history into one chronological timeline.
    Each unique fight appears once (root version kept when fight appears in both sources).
    Sorted oldest-first for ELO computation — future fights must never influence past ratings.
    Args:
        fights: pl.DataFrame of root fights with fighter1_id, fighter2_id, winner_id, fight_date
        prior_fights: pl.DataFrame of historical prior fights to include in the timeline
    Returns:
        pl.DataFrame with one row per unique fight, columns: fight_id, fighter1_id, fighter2_id,
        winner_id, fight_date, is_root (bool)
    """
    root = fights.select([
        pl.col('root_fight_id').alias('fight_id'),
        pl.col('fighter1_id'), pl.col('fighter2_id'), pl.col('winner_id'),
        pl.col('fight_date'), pl.lit(True).alias('is_root'),
    ])

    id_map = fights.select(['root_fight_id', 'fighter1_id', 'fighter2_id'])
    prior = (
        prior_fights
        .join(id_map, on='root_fight_id', how='left')
        .with_columns(
            pl.when(pl.col('fighter_role') == 'f1')
            .then(pl.col('fighter1_id')).otherwise(pl.col('fighter2_id')).alias('fighter_id')
        )
        .unique(subset=['prior_fight_id'], keep='first')
        .with_columns(
            pl.when(pl.col('result') == 'win')
            .then(pl.col('fighter_id')).otherwise(pl.col('opponent_id')).alias('winner_id')
        )
        .select([
            pl.col('prior_fight_id').alias('fight_id'),
            pl.col('fighter_id').alias('fighter1_id'),
            pl.col('opponent_id').alias('fighter2_id'),
            pl.col('winner_id'), pl.col('fight_date'),
            pl.lit(False).alias('is_root'),
        ])
    )

    return (
        pl.concat([root, prior])
        .sort(['fight_date', 'is_root'], descending=[False, True])
        .unique(subset=['fight_id'], keep='first')
        .sort(['fight_date', 'fight_id'])
    )


timeline = assemble_timeline(fights_df, prior_fights_df)
print('Timeline rows:', len(timeline), '  (root fights + all prior history)')
timeline.head(3)

## 4. ELO Computation Pass

One chronological sweep through the timeline, updating ratings after every fight. For each root fight we record a snapshot of pre-fight ELOs — these become the ML features.

**Peak ELO logic:** Each fighter has a "peak window" — a continuous stretch with no long layoff and no significant loss streak. The window resets (and peak restarts from current ELO) when a fighter is inactive for >1 year OR loses 2 consecutive fights. This prevents stale historical highs from inflating current predictions.

In [ ]:
PEAK_LAYOFF_DAYS = 365
PEAK_LOSS_STREAK = 2


def run_elo(timeline: pl.DataFrame) -> tuple:
    """
    Runs a single chronological ELO pass with variable K and peak-window tracking.
    Processes fights in date order — each fight updates ELO ratings before the next fight.
    Peak window resets after a layoff >PEAK_LAYOFF_DAYS or PEAK_LOSS_STREAK consecutive losses.
    Args:
        timeline: pl.DataFrame from assemble_timeline(), sorted oldest-first
    Returns:
        tuple of (final_elo_dict, root_snapshots_list) where:
          final_elo_dict is {fighter_id: final_elo} for all fighters
          root_snapshots_list is [{root_fight_id, f1_elo, f2_elo, f1_peak_elo, f2_peak_elo, winner_is_f1}]
    """
    elo:             dict = {}
    fight_count:     dict = {}
    consec_losses:   dict = {}
    last_fight_date: dict = {}
    window_peak:     dict = {}
    snapshots:       list = []

    for row in timeline.iter_rows(named=True):
        f1_id  = row['fighter1_id']
        f2_id  = row['fighter2_id']
        winner = row['winner_id']
        fid    = row['fight_id']
        fd     = row['fight_date']

        r1 = elo.get(f1_id, INITIAL_ELO)
        r2 = elo.get(f2_id, INITIAL_ELO)

        # Update peak windows before the fight (no leakage)
        for fp, curr in ((f1_id, r1), (f2_id, r2)):
            reset = (
                consec_losses.get(fp, 0) >= PEAK_LOSS_STREAK
                or (fp in last_fight_date and (fd - last_fight_date[fp]).days > PEAK_LAYOFF_DAYS)
            )
            if reset:
                window_peak[fp] = curr
                consec_losses[fp] = 0
            elif fp not in window_peak:
                window_peak[fp] = curr
            else:
                window_peak[fp] = max(window_peak[fp], curr)

        if row['is_root']:
            snapshots.append({
                'root_fight_id': fid,
                'f1_elo': r1, 'f2_elo': r2,
                'f1_peak_elo': window_peak[f1_id],
                'f2_peak_elo': window_peak[f2_id],
                'winner_is_f1': winner == f1_id,
            })

        s1 = 1.0 if winner == f1_id else 0.0
        K1 = get_k(fight_count.get(f1_id, 0))
        K2 = get_k(fight_count.get(f2_id, 0))

        elo[f1_id], _ = elo_update(r1, r2, s1, K1)
        _, elo[f2_id] = elo_update(r1, r2, s1, K2)

        fight_count[f1_id] = fight_count.get(f1_id, 0) + 1
        fight_count[f2_id] = fight_count.get(f2_id, 0) + 1

        if winner == f1_id:
            consec_losses[f1_id] = 0
            consec_losses[f2_id] = consec_losses.get(f2_id, 0) + 1
        else:
            consec_losses[f1_id] = consec_losses.get(f1_id, 0) + 1
            consec_losses[f2_id] = 0

        last_fight_date[f1_id] = fd
        last_fight_date[f2_id] = fd

    return elo, snapshots


final_elo, snapshots = run_elo(timeline)
print(f'Processed {len(final_elo)} unique fighters across {len(timeline)} fights')
print(f'Root fight snapshots: {len(snapshots)}')

## 5. Calibration Check

We validate that the ELO system is well-calibrated: fighters with a large ELO advantage should win more often than the model predicts, and vice versa.

In [ ]:
def calibration_loss(snapshots: list, min_elo_diff: float = 50.0) -> float:
    """
    Computes bucket-based mean absolute error between predicted and actual win rates.
    Splits fights into 5 quintiles by ELO difference and compares median predicted
    win probability to actual win rate per bucket. Uses median (not mean) on predicted
    prob to avoid extreme ELO gaps dominating the metric.
    Only includes fights where |elo_diff| > min_elo_diff to skip cold-start noise.
    Args:
        snapshots: list of dicts with f1_elo, f2_elo, winner_is_f1 keys
        min_elo_diff: minimum ELO difference to include a fight in calibration (default 50)
    Returns:
        float calibration MAE (lower is better; 0 = perfectly calibrated)
    """
    items = [
        (
            s['f1_elo'] - s['f2_elo'],
            1.0 / (1.0 + 10.0 ** ((s['f2_elo'] - s['f1_elo']) / 400.0)),
            1.0 if s['winner_is_f1'] else 0.0,
        )
        for s in snapshots
        if abs(s['f1_elo'] - s['f2_elo']) > min_elo_diff
    ]
    if not items:
        return float('inf')

    items.sort(key=lambda x: x[0])
    n = len(items)
    bin_size = max(n // 5, 1)

    total_err, n_bins = 0.0, 0
    bucket_data = []
    for i in range(5):
        chunk = items[i * bin_size : (i + 1) * bin_size if i < 4 else n]
        if not chunk:
            continue
        median_pred  = statistics.median(x[1] for x in chunk)
        actual_rate  = sum(x[2] for x in chunk) / len(chunk)
        total_err   += abs(median_pred - actual_rate)
        n_bins      += 1
        bucket_data.append((statistics.median(x[0] for x in chunk), median_pred, actual_rate))

    print('ELO Calibration by bucket:')
    print(f'{"ELO Diff":>10}  {"Predicted":>10}  {"Actual":>10}')
    for elo_diff, pred, actual in bucket_data:
        print(f'{elo_diff:>10.1f}  {pred:>10.2%}  {actual:>10.2%}')

    return total_err / n_bins if n_bins else float('inf')


cal_loss = calibration_loss(snapshots)
print(f'\nCalibration MAE: {cal_loss:.4f}  (lower is better)')

## 6. Top Fighters by Final ELO

In [ ]:
def print_top_fighters(final_elo: dict, fights: pl.DataFrame, n: int = 15) -> None:
    """
    Prints the top-N fighters ranked by their final ELO rating.
    Args:
        final_elo: dict of {fighter_id: elo_rating} after all fights are processed
        fights: pl.DataFrame with fighter1_id, fighter1_name, fighter2_id, fighter2_name columns
        n: number of fighters to display (default 15)
    Returns:
        None (prints ranking table to stdout)
    """
    names = {}
    for row in fights.select(['fighter1_id', 'fighter1_name', 'fighter2_id', 'fighter2_name']).iter_rows(named=True):
        names[row['fighter1_id']] = row['fighter1_name']
        names[row['fighter2_id']] = row['fighter2_name']

    ranked = sorted(final_elo.items(), key=lambda x: x[1], reverse=True)[:n]
    print(f'\n{"Rank":<6} {"Fighter":<28} {"ELO":>7}')
    print('-' * 43)
    for i, (fid, rating) in enumerate(ranked, 1):
        print(f'{i:<6} {names.get(fid, fid):<28} {rating:>7.1f}')


print_top_fighters(final_elo, fights_df)

## 7. ELO Distribution

We expect ELO ratings to follow a roughly normal distribution centered around 1500, with elite fighters pulling significantly above and weak/new fighters clustering near the initial value.

In [ ]:
elo_values = list(final_elo.values())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(elo_values, bins=60, color='steelblue', edgecolor='none', alpha=0.8)
axes[0].axvline(INITIAL_ELO, color='red', linestyle='--', label='Initial ELO (1500)')
axes[0].set_xlabel('ELO Rating')
axes[0].set_ylabel('Number of Fighters')
axes[0].set_title('Final ELO Distribution')
axes[0].legend()
for spine in axes[0].spines.values(): spine.set_visible(False)

# ELO over time (sampled from snapshots)
f1_elos = [s['f1_elo'] for s in snapshots]
axes[1].plot(f1_elos, alpha=0.3, linewidth=0.5, color='steelblue')
axes[1].axhline(INITIAL_ELO, color='red', linestyle='--', label='Initial ELO')
axes[1].set_xlabel('Fight index (chronological)')
axes[1].set_ylabel('F1 pre-fight ELO')
axes[1].set_title('F1 ELO Over Time (all root fights)')
axes[1].legend()
for spine in axes[1].spines.values(): spine.set_visible(False)

plt.tight_layout()
plt.show()

elo_df_snap = pl.DataFrame(snapshots)
print('\nELO feature statistics at root fights:')
print(elo_df_snap.select(['f1_elo', 'f2_elo', 'f1_peak_elo', 'f2_peak_elo']).describe())

## 8. ELO as a Predictive Feature

How well does raw ELO difference predict fight outcomes compared to random chance?

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score

elo_diff    = [s['f1_elo'] - s['f2_elo'] for s in snapshots]
winner_is_f1 = [1 if s['winner_is_f1'] else 0 for s in snapshots]

# Convert ELO diff to win probability for ROC-AUC
elo_proba = [1.0 / (1.0 + 10.0 ** (-d / 400.0)) for d in elo_diff]
elo_pred  = [1 if p >= 0.5 else 0 for p in elo_proba]

auc = roc_auc_score(winner_is_f1, elo_proba)
acc = accuracy_score(winner_is_f1, elo_pred)

print(f'ELO-only ROC-AUC : {auc:.4f}')
print(f'ELO-only Accuracy: {acc:.2%}')
print()
print('For comparison, the final model achieved:')
print('  Logistic Regression ROC-AUC : 0.6348  (58.73% accuracy)')
print('  LightGBM            ROC-AUC : 0.6319  (58.05% accuracy)')

## Summary

The ELO system demonstrates that a single principled rating captures meaningful signal about fighter quality. However, the AUC is lower than the final model's because:

1. **ELO ignores physical attributes** — age, reach, and weight are strong predictors that ELO does not capture.
2. **Cold-start problem** — Every fighter starts at 1500, so early fights look like even matchups regardless of true ability.
3. **UFC unpredictability** — A single KO or submission ends a fight regardless of the overall quality gap.

Adding ELO as a feature *on top of* the main notebook's features could push performance further. Future work could also explore per-weight-class ELO and domain ELOs (striking vs. grappling) as described in the notebook intro.